# Phase B.2.1 — LSI + Triangulation on Gemma 2 9B (layers 20 and 31)

**Prerequisite:** the smoke test `exp_gemma9b_smoke_test.ipynb` PASSED (185+ robust PT-dominant features at L20).

**Goal:** Replicate the distributive part of the 2B pipeline on 9B, across two layers (L20 ≈ 48%, L31 ≈ 74%), with Wikipedia + FineWeb triangulation.

**Setup:**
- Model: `google/gemma-2-9b` (42 layers, $d_\text{model} = 3584$)
- SAEs: `gemma-scope-9b-pt-res-canonical/layer_{20,31}/width_16k/canonical`
- Corpora: Wikipedia PT/EN (20231101) + FineWeb-2 PT / FineWeb EN, 1M tokens each
- Expected VRAM: ~18.5 GB in bfloat16 (fits on an L4 22 GB with headroom)

**Outputs:**
- `gemma9b_lsi_layer20_31.json` — counts, freqs, LSI, triangulation per layer
- `gemma9b_lsi_layer20_31_counts.pt` — raw tensors (counts per layer, corpus)

**Next stages:**
- B.2.2 `exp_gemma9b_probes.ipynb` — uses the candidate features from this JSON.
- B.2.3 `exp_gemma9b_steering.ipynb` — uses the candidates validated in B.2.2.

## 1. Instalação e ambiente

In [ ]:
!pip install -q sae-lens transformer-lens datasets

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:    {name} ({total_gb:.1f} GB total)")
    assert total_gb >= 20, "Precisa de >=20GB VRAM em bf16. Use L4 ou A100."
else:
    raise RuntimeError("Pipeline exige GPU.")

## 2. Configuração

In [ ]:
MODEL_NAME = "gemma-2-9b"
HF_MODEL_ID = f"google/{MODEL_NAME}"
LAYERS = [20, 31]
WIDTH = "16k"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"

N_TOKENS = 1_000_000
SEQ_LEN = 256
BATCH_SIZE = 4

DTYPE = torch.bfloat16
MIN_ACTS = 50
THRESH_PT = 0.30
THRESH_EN = -0.30
THRESH_CROSS = 0.10

OUT_JSON   = "gemma9b_lsi_layer20_31.json"
OUT_COUNTS = "gemma9b_lsi_layer20_31_counts.pt"

n_seqs = N_TOKENS // SEQ_LEN
n_batches = n_seqs // BATCH_SIZE
print(f"Tokens/corpus: {N_TOKENS:,} | seqs: {n_seqs} | batches: {n_batches} (batch={BATCH_SIZE}, seq={SEQ_LEN})")
print(f"Layers:        {LAYERS}")
print(f"SAE release:   {SAE_RELEASE}")

## 3. Load model and SAEs

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.empty_cache()
gc.collect()

print(f"Carregando {HF_MODEL_ID} em {DTYPE} (~4-6 min na L4)...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": device},
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

n_model_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"layers: {n_model_layers} | d_model: {d_model}")
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
from sae_lens import SAE

saes = {}
for L in LAYERS:
    sae_id = f"layer_{L}/width_{WIDTH}/canonical"
    print(f"Carregando SAE {sae_id}...")
    s = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=device)
    if isinstance(s, tuple):
        s = s[0]
    s = s.to(DTYPE)
    saes[L] = s
    print(f"  d_sae={s.cfg.d_sae} | d_in={s.cfg.d_in}")
    assert s.cfg.d_in == d_model, f"d_in {s.cfg.d_in} != d_model {d_model}"

D_SAE = saes[LAYERS[0]].cfg.d_sae
for L in LAYERS:
    assert saes[L].cfg.d_sae == D_SAE
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB | d_sae={D_SAE}")

## 4. Multi-layer hooks

We register a forward hook on each target block. In a single pass through the model, we capture the residual-stream activations at both layers and encode each with the corresponding SAE.

In [ ]:
_cap = {}
_handles = []

def _make_hook(layer_idx):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        _cap[layer_idx] = h
    return hook

def install_hooks(layers):
    for h in _handles:
        h.remove()
    _handles.clear()
    _cap.clear()
    for L in layers:
        handle = model.model.layers[L].register_forward_hook(_make_hook(L))
        _handles.append(handle)

def encode_batch(batch_ids, layers):
    """batch_ids: (B, T) on device. Returns dict layer -> (B, T, d_sae)."""
    _cap.clear()
    with torch.no_grad():
        model(input_ids=batch_ids, use_cache=False)
    feats = {}
    for L in layers:
        resid = _cap[L]
        feats[L] = saes[L].encode(resid)
    return feats

install_hooks(LAYERS)

test_ids = tokenizer("O Brasil é o maior país da América do Sul.", return_tensors="pt").input_ids.to(device)
test_feats = encode_batch(test_ids, LAYERS)
for L in LAYERS:
    l0 = (test_feats[L] > 0).float().sum(dim=-1).mean().item()
    print(f"Layer {L}: shape={tuple(test_feats[L].shape)} | L0 médio={l0:.1f}")
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 5. Collecting the 4 corpora (1M tokens each)

Wikipedia PT (Level 2) + Wikipedia EN (Level 2) + FineWeb-2 PT (Level 1) + FineWeb EN (Level 1).

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

TEXT_KEYS = {"wiki": "text", "fineweb": "text"}

def collect_tokens(stream, text_key, n_tokens, desc):
    all_ids = []
    n_docs = 0
    pbar = tqdm(total=n_tokens, desc=desc, unit="tok")
    for doc in stream:
        text = doc.get(text_key) or ""
        if not text:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_ids.extend(ids)
        n_docs += 1
        pbar.update(len(ids))
        if len(all_ids) >= n_tokens:
            break
    pbar.close()
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_docs} docs -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs x {SEQ_LEN})")
    return tokens

CORPORA = {}

print("[1/4] Wikipedia PT...")
ds = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
CORPORA["wiki_pt"] = collect_tokens(ds, "text", N_TOKENS, "wiki_pt")

print("[2/4] Wikipedia EN...")
ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
CORPORA["wiki_en"] = collect_tokens(ds, "text", N_TOKENS, "wiki_en")

print("[3/4] FineWeb-2 PT...")
ds = load_dataset("HuggingFaceFW/fineweb-2", "por_Latn", split="train", streaming=True)
CORPORA["fineweb_pt"] = collect_tokens(ds, "text", N_TOKENS, "fineweb_pt")

print("[4/4] FineWeb EN...")
ds = load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True)
CORPORA["fineweb_en"] = collect_tokens(ds, "text", N_TOKENS, "fineweb_en")

for name, t in CORPORA.items():
    print(f"  {name}: {t.shape} = {t.numel():,} tokens")

## 6. Activation extraction: counts per (corpus, layer, feature)

For each corpus we run the model once per batch and capture the residual stream at both layers in a single pass. For each feature we count in how many tokens it is active (>0).

In [ ]:
import os

def feature_counts_multilayer(tokens, layers, batch_size, desc):
    counts = {L: torch.zeros(D_SAE, dtype=torch.float32, device=device) for L in layers}
    total_tokens = 0
    n_full = (len(tokens) // batch_size) * batch_size
    for i in tqdm(range(0, n_full, batch_size), desc=desc):
        batch = tokens[i:i+batch_size].to(device)
        feats = encode_batch(batch, layers)
        for L in layers:
            counts[L] += (feats[L] > 0).float().sum(dim=(0, 1))
        total_tokens += batch.numel()
        del feats
        if (i // batch_size) % 32 == 0:
            torch.cuda.empty_cache()
    return {L: counts[L].cpu() for L in layers}, total_tokens

COUNTS = {}
TOK_TOTALS = {}
for name, tokens in CORPORA.items():
    print(f"\n=== {name} ===")
    counts, total = feature_counts_multilayer(tokens, LAYERS, BATCH_SIZE, name)
    COUNTS[name] = counts
    TOK_TOTALS[name] = total
    print(f"  tokens processados: {total:,}")
    torch.save({"COUNTS": COUNTS, "TOK_TOTALS": TOK_TOTALS, "LAYERS": LAYERS, "D_SAE": D_SAE}, OUT_COUNTS)
    print(f"  checkpoint salvo em {OUT_COUNTS}")

## 7. LSI por nível e triangulação

In [ ]:
eps = 1e-10

def to_freq(counts, total):
    return counts / max(total, 1)

FREQS = {name: {L: to_freq(COUNTS[name][L], TOK_TOTALS[name]) for L in LAYERS} for name in CORPORA}

LSI = {"wiki": {}, "fineweb": {}}
for L in LAYERS:
    LSI["wiki"][L]    = (FREQS["wiki_pt"][L]    - FREQS["wiki_en"][L])    / (FREQS["wiki_pt"][L]    + FREQS["wiki_en"][L]    + eps)
    LSI["fineweb"][L] = (FREQS["fineweb_pt"][L] - FREQS["fineweb_en"][L]) / (FREQS["fineweb_pt"][L] + FREQS["fineweb_en"][L] + eps)

ACTIVE = {}
for L in LAYERS:
    ACTIVE[L] = (
        (COUNTS["wiki_pt"][L]    + COUNTS["wiki_en"][L]    >= MIN_ACTS) &
        (COUNTS["fineweb_pt"][L] + COUNTS["fineweb_en"][L] >= MIN_ACTS)
    )
    print(f"Layer {L}: features ativas em ambos níveis = {int(ACTIVE[L].sum())} / {D_SAE}")

In [ ]:
def classify(layer):
    w = LSI["wiki"][layer]
    f = LSI["fineweb"][layer]
    act = ACTIVE[layer]
    pt_both = act & (w > THRESH_PT) & (f > THRESH_PT)
    en_both = act & (w < THRESH_EN) & (f < THRESH_EN)
    cross_both = act & (w.abs() < THRESH_CROSS) & (f.abs() < THRESH_CROSS)
    return pt_both, en_both, cross_both

CLASS = {}
for L in LAYERS:
    pt_b, en_b, cr_b = classify(L)
    CLASS[L] = {"pt_both": pt_b, "en_both": en_b, "cross_both": cr_b}
    mean_lsi = (LSI["wiki"][L][ACTIVE[L]].mean() + LSI["fineweb"][L][ACTIVE[L]].mean()) / 2
    print(f"\nLayer {L}:")
    print(f"  PT-both    (LSI>{THRESH_PT} em wiki e fineweb): {int(pt_b.sum()):>6}")
    print(f"  EN-both    (LSI<{THRESH_EN} em wiki e fineweb): {int(en_b.sum()):>6}")
    print(f"  Cross-both (|LSI|<{THRESH_CROSS} em wiki e fineweb): {int(cr_b.sum()):>6}")
    print(f"  Mean LSI (active, média wiki+fineweb): {mean_lsi.item():+.4f}")

## 8. Top features PT-both e EN-both por camada

In [ ]:
TOP_K = 20

def rank_pt_both(layer, k=TOP_K):
    w = LSI["wiki"][layer]
    f = LSI["fineweb"][layer]
    score = torch.minimum(w, f)
    score = score.clone()
    score[~CLASS[layer]["pt_both"]] = -float("inf")
    vals, idx = score.topk(k)
    return idx, vals

def rank_en_both(layer, k=TOP_K):
    w = LSI["wiki"][layer]
    f = LSI["fineweb"][layer]
    score = torch.maximum(w, f)
    score = score.clone()
    score[~CLASS[layer]["en_both"]] = float("inf")
    vals, idx = (-score).topk(k)
    return idx, -vals

TOP_PT = {}
TOP_EN = {}
for L in LAYERS:
    pt_idx, _ = rank_pt_both(L)
    en_idx, _ = rank_en_both(L)
    TOP_PT[L] = pt_idx.tolist()
    TOP_EN[L] = en_idx.tolist()
    print(f"\nLayer {L} — Top-{TOP_K} PT-both (ordenado por min(LSI_wiki, LSI_fineweb)):")
    print(f"  {'feat':>6} {'LSI_w':>8} {'LSI_f':>8} {'f_pt_w':>10} {'f_en_w':>10} {'f_pt_f':>10} {'f_en_f':>10}")
    for fid in TOP_PT[L][:10]:
        print(f"  {fid:>6} "
              f"{LSI['wiki'][L][fid].item():>+8.3f} "
              f"{LSI['fineweb'][L][fid].item():>+8.3f} "
              f"{FREQS['wiki_pt'][L][fid].item():>10.5f} "
              f"{FREQS['wiki_en'][L][fid].item():>10.5f} "
              f"{FREQS['fineweb_pt'][L][fid].item():>10.5f} "
              f"{FREQS['fineweb_en'][L][fid].item():>10.5f}")

## 9. Comparação cross-layer (L20 vs L31)

In [ ]:
pt_b_20 = CLASS[20]["pt_both"]
pt_b_31 = CLASS[31]["pt_both"]
overlap = (pt_b_20 & pt_b_31).sum().item()
only_20 = (pt_b_20 & ~pt_b_31).sum().item()
only_31 = (pt_b_31 & ~pt_b_20).sum().item()
print(f"PT-both em L20:        {int(pt_b_20.sum())}")
print(f"PT-both em L31:        {int(pt_b_31.sum())}")
print(f"Em ambas (overlap):    {overlap}")
print(f"Só em L20:             {only_20}")
print(f"Só em L31:             {only_31}")
print("\nNote: SAEs from different layers are independent, so the \"overlap\" aqui")
print("refers to features with the SAME feature_id in both SAEs. There is no guarantee")
print("of semantic correspondence between features of identical feature_id across layers")
print("distintas; isto é apenas um indicador agregado.")

## 10. Comparação 2B (referência) vs 9B

Valores do 2B vêm da Tabela do paper (multi-layer LSI com 1M tokens/corpus). Comparamos contagens absolutas e LSI médio.

In [ ]:
REFS_2B = {
    "layer_5":  {"pt_both": 1415, "mean_lsi": -0.031, "depth_pct": 19.2},
    "layer_9":  {"pt_both": 1224, "mean_lsi": -0.171, "depth_pct": 34.6},
    "layer_13": {"pt_both":  651, "mean_lsi": -0.322, "depth_pct": 50.0},
    "layer_17": {"pt_both":  768, "mean_lsi": -0.355, "depth_pct": 65.4},
    "layer_21": {"pt_both": 1058, "mean_lsi": -0.320, "depth_pct": 80.8},
}

print("== 2B (referência, 1M tokens/corpus, 16k SAE) ==")
print(f"  {'layer':<10} {'depth%':>8} {'PT-both':>10} {'mean_LSI':>10}")
for k, v in REFS_2B.items():
    print(f"  {k:<10} {v['depth_pct']:>8.1f} {v['pt_both']:>10} {v['mean_lsi']:>+10.3f}")

print("\n== 9B (este experimento) ==")
print(f"  {'layer':<10} {'depth%':>8} {'PT-both':>10} {'mean_LSI':>10}")
for L in LAYERS:
    mean_lsi = (LSI["wiki"][L][ACTIVE[L]].mean() + LSI["fineweb"][L][ACTIVE[L]].mean()) / 2
    depth = 100 * L / n_model_layers
    print(f"  layer_{L:<4} {depth:>8.1f} {int(CLASS[L]['pt_both'].sum()):>10} {mean_lsi.item():>+10.3f}")

## 11. Save results

In [ ]:
import json

def top_records(layer, idx_list):
    out = []
    for fid in idx_list:
        out.append({
            "feature_id": int(fid),
            "lsi_wiki": float(LSI["wiki"][layer][fid].item()),
            "lsi_fineweb": float(LSI["fineweb"][layer][fid].item()),
            "freq_pt_wiki": float(FREQS["wiki_pt"][layer][fid].item()),
            "freq_en_wiki": float(FREQS["wiki_en"][layer][fid].item()),
            "freq_pt_fineweb": float(FREQS["fineweb_pt"][layer][fid].item()),
            "freq_en_fineweb": float(FREQS["fineweb_en"][layer][fid].item()),
            "count_pt_wiki": int(COUNTS["wiki_pt"][layer][fid].item()),
            "count_en_wiki": int(COUNTS["wiki_en"][layer][fid].item()),
            "count_pt_fineweb": int(COUNTS["fineweb_pt"][layer][fid].item()),
            "count_en_fineweb": int(COUNTS["fineweb_en"][layer][fid].item()),
        })
    return out

result = {
    "experiment": "gemma9b_lsi_multilayer",
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "sae_width": WIDTH,
    "layers": LAYERS,
    "n_model_layers": n_model_layers,
    "d_sae": D_SAE,
    "d_model": d_model,
    "n_tokens_per_corpus": N_TOKENS,
    "min_acts": MIN_ACTS,
    "thresholds": {"pt": THRESH_PT, "en": THRESH_EN, "cross": THRESH_CROSS},
    "tokens_processed": {name: int(t) for name, t in TOK_TOTALS.items()},
    "by_layer": {},
    "cross_layer": {
        "pt_both_overlap_l20_l31": int(overlap),
        "pt_both_only_l20": int(only_20),
        "pt_both_only_l31": int(only_31),
    },
    "refs_2b": REFS_2B,
}

for L in LAYERS:
    mean_lsi_wiki    = LSI["wiki"][L][ACTIVE[L]].mean().item()
    mean_lsi_fineweb = LSI["fineweb"][L][ACTIVE[L]].mean().item()
    result["by_layer"][str(L)] = {
        "layer": L,
        "depth_pct": round(100 * L / n_model_layers, 2),
        "n_active_both": int(ACTIVE[L].sum()),
        "n_pt_both": int(CLASS[L]["pt_both"].sum()),
        "n_en_both": int(CLASS[L]["en_both"].sum()),
        "n_cross_both": int(CLASS[L]["cross_both"].sum()),
        "mean_lsi_wiki": float(mean_lsi_wiki),
        "mean_lsi_fineweb": float(mean_lsi_fineweb),
        "mean_lsi_avg": float((mean_lsi_wiki + mean_lsi_fineweb) / 2),
        "top20_pt_both": top_records(L, TOP_PT[L]),
        "top20_en_both": top_records(L, TOP_EN[L]),
    }

with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Salvo em {OUT_JSON}")
print(f"Counts brutos em {OUT_COUNTS}")
for L in LAYERS:
    info = result["by_layer"][str(L)]
    print(f"  L{L}: PT-both={info['n_pt_both']} EN-both={info['n_en_both']} "
          f"cross-both={info['n_cross_both']} mean_LSI={info['mean_lsi_avg']:+.3f}")

## 12. Next steps

**If PT-both > 100 in at least one layer**: proceed to B.2.2 (gender + crase + clitics probes on 9B) using this JSON as a starting point.

**If PT-both is too low**:
- Check the mean L0 in cell 4 (should be in the hundreds; if 0, the hook is wrong).
- Check tokens processed across all corpora (should be close to 1M each).
- Try another layer (L9 or L24).

**To reproduce without running from scratch**: the checkpoint `gemma9b_lsi_layer20_31_counts.pt` contains the raw counts and allows recomputing LSI/triangulation without reprocessing 4M tokens.